# NB63: SIEM Log Analysis

Kafka -> Spark -> ES/MinIO

## 1. Environment Setup

Installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and Python libraries (PySpark, Kafka-Python, Redis, Mongo, ES, Cassandra, MinIO).

In [ ]:
# Install Dependencies (Java 8, Spark 3.5.0, Kafka 3.6.1)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio "numpy<2.0.0"

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

Starts background services needed for this pipeline:
- **Kafka** (Zookeeper + Broker)
- **Elasticsearch**
- **MinIO**

In [ ]:
# Start Kafka
!!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Elasticsearch
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.10.2-linux-x86_64.tar.gz
!tar -xzf elasticsearch-7.10.2-linux-x86_64.tar.gz
!chown -R daemon:daemon elasticsearch-7.10.2
!!sudo -u daemon ES_JAVA_OPTS="-Xms512m -Xmx512m" ./elasticsearch-7.10.2/bin/elasticsearch -d
# Start MinIO
!wget -q https://dl.min.io/server/minio/release/linux-amd64/minio
!chmod +x minio
!./minio server /data --console-address ":9001" &> minio.log &

import time, socket
def wait_for_port(port, host='localhost', timeout=60):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port}")
                return False
            time.sleep(2)

# Wait for services
if 'kafka' in locals() or 'kafka' in globals(): wait_for_port(9092) # Kafka
wait_for_port(9042) # Cassandra
time.sleep(10) # Extra buffer for Cassandra to be fully ready

## 3. Create Kafka Topic

Creates a topic named `input-topic`.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer (Security Logs)

Simulates log events with levels INFO/WARN/ERROR.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting Security Log Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 500 logs...")
levels = ['INFO', 'WARN', 'ERROR']
for _ in range(500):
    log = {'timestamp': time.time(), 'level': random.choice(levels), 'msg': 'Activity detected'}
    producer.send('input-topic', json.dumps(log).encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Security Event Pipeline

1. Filters ERROR logs for immediate indexing in Elasticsearch (`alerts`).
2. (Placeholder) Could archive other logs to MinIO.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
from elasticsearch import Elasticsearch
import json

spark = SparkSession.builder.appName("SIEM").getOrCreate()

def process_batch(df, epoch_id):
    rows = df.collect()
    es = Elasticsearch(['http://localhost:9200'])
    for row in rows:
        log = json.loads(row.value)
        if log['level'] == 'ERROR':
            # Hot Storage
            es.index(index='alerts', body=log)
    print(f"Batch {epoch_id} processed: Analyzed security events.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Count Alerts in Elasticsearch.

In [ ]:
from elasticsearch import Elasticsearch
es = Elasticsearch(['http://localhost:9200'])
time.sleep(2)
try:
    res = es.count(index="alerts")
    print(f"Alerts Found: {res['count']}")
except: print("No alerts found yet.")